# Engagement Model V5 — runnable notebook

This is a thin notebook wrapper around the `engagement_model` Python package
(refactored from the original research notebook into `src/engagement_model/`).

All the modeling logic (K-hop social graph, ResNet/time-attention encoders,
Bi-LSTM skeleton encoder + FiLM conditioning, GNN + multimodal fusion
transformer, multi-task/ordinal/contrastive losses) lives in the package; this
notebook only wires it together so you can run it interactively on Kaggle,
Colab, or any Jupyter environment, and inspect intermediate results.

**Before running:** edit the `CONFIG_OVERRIDES` dict in the second code cell
so `OUTPUT_DIR`, `FEATURE_DIR`, `SKELETON_DIR`, `SPLIT_NAMES_JSON_PATH` and
`CHECKPOINT_DIR` point at your data/checkpoint locations.

In [ ]:
# If the package isn't installed yet, install it in editable mode.
# On Kaggle/Colab, replace the path below with wherever you uploaded this repo.
import sys, os

REPO_ROOT = os.path.abspath("..")  # this notebook lives in <repo>/notebooks/
SRC_PATH = os.path.join(REPO_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import engagement_model
print("engagement_model package version:", engagement_model.__version__)

## 1. Configuration

In [ ]:
CONFIG_OVERRIDES = {
    # ---- EDIT THESE PATHS for your environment ----
    "OUTPUT_DIR": "/kaggle/input/datasets/deadwish1/engagement-dataset-npy",
    "FEATURE_DIR": "/kaggle/input/datasets/drakhight/resnet-engagement/resnet_features",
    "SKELETON_DIR": "/kaggle/input/datasets/drakhight/skeleton-engagement/skeleton_features_yolo/skeleton",
    "SPLIT_NAMES_JSON_PATH": "/kaggle/input/datasets/deadwish1/slip-names/split_names.json",
    "CHECKPOINT_DIR": "/kaggle/working/checkpoints_v5",

    # Set True for a quick end-to-end smoke test (tiny subset, 2 epochs) before a full run.
    "DRY_RUN": False,
}

## 2. Run everything with one call

`run_full_pipeline` runs the exact same sequence as the original notebook:
manifest loading -> diagnostics -> K-hop graph / context window -> cache
filtering -> DataLoaders -> model/optimizer/scheduler -> training with early
stopping -> checkpoint selection -> test evaluation (bias tuning, session-level
aggregation, confusion matrix, uncertainty/risk-coverage, social diagnostics).

If you'd rather run it step by step (e.g. to inspect the dataframe or the
neighbor graph before training), see section 3 below instead.

In [ ]:
from engagement_model.pipeline import run_full_pipeline

result = run_full_pipeline(config_overrides=CONFIG_OVERRIDES, run_diagnostics=True)

print("Best checkpoint:", result["train_result"]["best_ckpt_path"])
print("Best val macro-F1:", result["train_result"]["best_val_macro_f1"])
if result["test_result"] is not None:
    print("Test macro-F1:", result["test_result"]["test_metrics"]["macro_f1"])

In [ ]:
# Training history as a DataFrame, handy for plotting.
result["train_result"]["history_df"]

## 3. Step-by-step run (optional)

Use this instead of section 2 if you want to inspect intermediate artifacts
(the manifest dataframe, the neighbor graph, a single batch, ...).

In [ ]:
from engagement_model.config import get_config, print_config, setup_device
from engagement_model.data.manifest import load_manifest_and_prepare, prepare_aux_labels
from engagement_model.data.diagnostics import (
    diagnose_track_split_leakage,
    diagnose_label_stability_within_track,
    diagnose_cache_dirs,
)
from engagement_model.data.graph import build_neighbor_index, build_context_window_index
from engagement_model.data.filtering import filter_by_cache_availability
from engagement_model.data.dataset import build_dataloaders
from engagement_model.losses import build_criterion
from engagement_model.training.setup import build_model, build_optimizer_and_scheduler, build_scaler
from engagement_model.training.engine import benchmark_pipeline
from engagement_model.training.loop import run_training
from engagement_model.evaluation.test_eval import run_test_evaluation

cfg = get_config(CONFIG_OVERRIDES)
print_config(cfg)
device, n_gpus = setup_device()

In [ ]:
df, label2id, id2label, num_classes = load_manifest_and_prepare(cfg)
df["engagement_label"].value_counts()

In [ ]:
_ = diagnose_track_split_leakage(df)
_ = diagnose_label_stability_within_track(df)
diagnose_cache_dirs(df, cfg)

In [ ]:
df, aux_label2id, aux_id2label = prepare_aux_labels(df, cfg)

neighbor_lists = build_neighbor_index(
    df, k_neighbors=cfg["K_NEIGHBORS"],
    candidate_multiplier=cfg["NEIGHBOR_CANDIDATE_MULTIPLIER"],
    require_time_overlap=cfg["NEIGHBOR_REQUIRE_TIME_OVERLAP"],
    skeleton_dir=cfg["SKELETON_DIR"], skeleton_conf_thr=cfg["SKELETON_CONF_THR"],
    use_orientation_penalty=cfg["USE_ORIENTATION_PENALTY"],
    orientation_angle_threshold_deg=cfg["ORIENTATION_ANGLE_THRESHOLD_DEG"],
    orientation_penalty=cfg["ORIENTATION_PENALTY"],
)
context_lists = build_context_window_index(
    df, window_size=cfg["CONTEXT_WINDOW_SIZE"],
    max_time_gap_seconds=cfg["CONTEXT_MAX_TIME_GAP_SECONDS"],
)

df, neighbor_lists, label2id, id2label, num_classes = filter_by_cache_availability(
    df, neighbor_lists, cfg
)

aux_task_names = list(cfg["AUX_LABEL_COLUMNS"].keys()) if cfg["USE_BEHAVIOR_EMOTION_AUX"] else []
cfg["_AUX_NUM_CLASSES"] = {t: len(aux_label2id.get(t, {})) for t in aux_task_names}

In [ ]:
loaders = build_dataloaders(df, cfg, label2id, aux_label2id)
criterion = build_criterion(cfg, df, label2id).to(device)
model = build_model(cfg, num_classes, device, n_gpus)
optimizer, scheduler = build_optimizer_and_scheduler(model, cfg, loaders["train_loader"])
scaler, amp_dtype, use_grad_scaler = build_scaler(cfg, device)

if cfg["RUN_SPEED_BENCHMARK"]:
    benchmark_pipeline(model, loaders["train_loader"], cfg, device, n_batches=cfg["BENCHMARK_BATCHES"])

In [ ]:
train_result = run_training(
    model, loaders["train_loader"], loaders["val_loader"],
    optimizer, scheduler, criterion, scaler, cfg, device,
    label2id, id2label, num_classes, aux_task_names,
)
train_result["history_df"]

In [ ]:
test_result = run_test_evaluation(
    cfg, device, n_gpus, num_classes,
    train_result["best_ckpt_path"],
    loaders["test_loader"], loaders["val_loader"], criterion,
    label2id, id2label, train_result["label_names_ordered"],
    aux_task_names, df,
    ordinal_rank_by_id=train_result["ordinal_rank_by_id"],
)
test_result["test_metrics"]["macro_f1"]